# 対話アプリの作成

## 参考

- [Adding MCP Tools to Reachy Mini (June 3 2026, Hugging Face)](https://huggingface.co/blog/adding-mcp-tools-to-reachy-mini)
    - [pollen-robotics/reachy-mini-search-tool (Hugging Face Spaces)](https://huggingface.co/spaces/pollen-robotics/reachy-mini-search-tool)
    - [pollen-robotics/reachy-mini-weather-tool (Hugging Face Spaces)](https://huggingface.co/spaces/pollen-robotics/reachy-mini-weather-tool)
- [Reachy Mini Tools (Hugging Face Spaces)](https://huggingface.co/spaces?q=reachy-mini&filter=reachy_mini)

## 概要

[pollen-robotics/reachy_mini_conversation_app](https://github.com/pollen-robotics/reachy_mini_conversation_app)を参考にOpenAIのAPIと連携したコンパニオンアプリを作る。

## 特徴

reachy_mini_conversation_appは、「Hey Reachy」と話しかけるとReachy Miniが顔を追従し、ダンスしたり、動きで感情を表現しながら、音声で応答するアプリ。Web UIで文字起こしの結果を確認できる。

- **低遅延のボイスループ**：リアルタイムモデルを使い、スムーズで高品質な音声会話ができる。
- **カメラによる状況把握（ビジョン対応）**：カメラツールで目の前のものを認識したり、顔を追跡したり、話している人に注目させたりできる。
- **豊かな表現のモーション**：会話中にダンスをキューに入れたり、録音済みの感情モーションを再生させたりできる。
- **オンデマンドなパーソナリティ**：Web UIからプロフィールを切り替えて会話スタイルを変更でき、キャラクターごとに使えるツール（ダンス・カメラ・トラッキングなど）を個別に設定できる。
- **柔軟なセットアップ**：有線・無線どちらのReachy Miniでも動作し、ビジョン処理はローカルまたはクラウドモデルのどちらでも実行できる。

## 仮想環境を構築

```sh
uv venv reachy_mini_conversation_app_env --python 3.12 --seed
```

In [ ]:
# リポジトリをクローン

import os

if not os.path.exists("reachy_mini_conversation_app"):
    !git clone https://github.com/pollen-robotics/reachy_mini_conversation_app.git

In [ ]:
# ソースコードをインストール
try:
    import reachy_mini_conversation_app
except ImportError:
    %pip install -e reachy_mini_conversation_app

In [ ]:
# 追加のオプションで視覚機能をインストール

# ローカルVLM（SmolVLM2）の実行に必要な依存関係。GPUが必要。
# %pip install -e .[local_vision]

# YOLOv11nのヘッドトラッキングに必要な依存関係。デフォルトでCPUで動き、GPUもサポート。
# %pip install -e .[yolo_vision]

# MediaPipeのランドマーク検出に必要な依存関係。軽量でCPUで動作。
# %pip install -e .[mediapipe_vision]

# MCPを介したHugging Face Spaceツールの実行に必要な依存関係。
# %pip install -e .[remote_tools]

# 全ての視覚機能の依存関係のインストール
# %pip install -e .[all_vision]

# pytest, ruff, mypyなどの開発ツールのインストール
# %pip install -e .[dev]

In [ ]:
%pip show reachy_mini_conversation_app # v0.7.0

## アーキテクチャ

[fastrtc](https://github.com/gradio-app/fastrtc)を使った低遅延ストリーミングによる、リアルタイム音声会話ループで動作する。

対応バックエンドは以下の通り。

- **Hugging Face**（デフォルト）：組み込みのHugging Faceサーバー、またはローカルエンドポイントを使用。
- **OpenAI Realtime**（`gpt-realtime-2`）：`OPENAI_API_KEY` が必要。
- **Gemini Live**（`gemini-3.1-flash-live-preview`）：`GEMINI_API_KEY` が必要。

画像処理はデフォルトで選択中のリアルタイムバックエンドを使用（カメラツール使用時）。`--local-vision` オプションでSmolVLM2（CPU/GPU/MPS）によるオンデバイス処理にも切り替えられる。

モーションはレイヤー構造で管理されており、主要な動き（ダンス・感情表現・ポーズ移動・呼吸）をキューに積みながら、発話連動のゆらぎやヘッドトラッキングをブレンドする。

非同期ツールディスパッチにより、ロボットモーション・カメラキャプチャ・ヘッドトラッキングをGradioのWeb UI（リアルタイム文字起こし付き）から統合的に操作できる。

![](asset/architecture.png)

## 環境変数の設定

`.env.example`に記載がある。以下の環境変数を設定できる。

| 変数名 | デフォルト | 説明 |
|--------|-----------|------|
| `OPENAI_API_KEY` | — | OpenAIのAPIキー（`openai` バックエンド使用時に必須） |
| `GEMINI_API_KEY` | — | GeminiのAPIキー（`gemini` バックエンド使用時に必須）。`GOOGLE_API_KEY` でも可。[aistudio.google.com](https://aistudio.google.com) で取得できる |
| `BACKEND_PROVIDER` | `huggingface` | 使用するバックエンド。`huggingface` / `openai` / `gemini` のいずれか |
| `MODEL_NAME` | （バックエンド依存） | モデル名の上書き。OpenAIは `gpt-realtime-2`、Geminiは `gemini-3.1-flash-live-preview` がデフォルト。Hugging Faceはサーバー側のモデル設定を使うため無視される |
| `REALTIME_TRANSCRIPTION_LANGUAGE` | `en` | 文字起こしの言語コード。バックエンドがサポートするコードを指定（例：`ja`、`zh`） |
| `HF_REALTIME_CONNECTION_MODE` | `deployed` | Hugging Faceの接続モード。`deployed`（公式サーバー）または `local`（`HF_REALTIME_WS_URL` を使用） |
| `HF_REALTIME_WS_URL` | — | 自前のHugging FaceバックエンドのWebSocketエンドポイント。ベースURL（`ws://127.0.0.1:8765/v1`）またはフルURL（`ws://127.0.0.1:8765/v1/realtime`）を指定。`HF_REALTIME_CONNECTION_MODE=local` 時に使用 |
| `HF_HOME` | `./cache` | Hugging Faceのローカルダウンロードキャッシュディレクトリ（`--local-vision` フラグ使用時のみ） |
| `HF_TOKEN` | — | Hugging Faceのアクセストークン（限定公開・プライベートアセットへのアクセス時に必要） |
| `LOCAL_VISION_MODEL` | `HuggingFaceTB/SmolVLM2-2.2B-Instruct` | ローカルビジョン処理に使うHugging FaceモデルのID（`--local-vision` フラグ使用時のみ） |
| `REACHY_MINI_CUSTOM_PROFILE` | — | 起動時に使用するプロフィール名（`profiles/` 内のフォルダ名） |
| `REACHY_MINI_EXTERNAL_PROFILES_DIRECTORY` | — | 外部プロフィールディレクトリのパス |
| `REACHY_MINI_EXTERNAL_TOOLS_DIRECTORY` | — | 外部ツールディレクトリのパス |
| `AUTOLOAD_EXTERNAL_TOOLS` | — | `1` にすると外部ツールディレクトリ内のツールを自動読み込み |

## プロファイル

`profiles/` フォルダに定義済みのキャラクタープロファイルが用意されている。`REACHY_MINI_CUSTOM_PROFILE` 環境変数またはWeb UIから切り替えられる。

| プロファイル名 | キャラクター |
|--------------|------------|
| `default` | デフォルト設定 |
| `bored_teenager` | 退屈なGen Zのティーン。小文字・一言・疲れたため息で返答する |
| `captain_circuit` | 海賊ロボット。「aye」「matey」を交えながら宝や海について語る |
| `chess_coach` | チェスコーチ。プレイヤーの手に対して自分の手を返しながら解説してくれる |
| `cosmic_kitchen` | キッチンに不時着した皮肉なロボット。火星探査機への憧れと食べ物ジョークを交える |
| `example` | カスタムツール（`sweep_look`）実装のサンプル。ロブスタージョーク好き |
| `hype_bot` | ハイエナジーコーチ。スポーツ比喩で15語以内の激励を叫ぶ |
| `mad_scientist_assistant` | 狂気の科学者アシスタント。「マスター」と呼びかけ、シューっとした一言で答える |
| `mars_rover` | 自分を火星探査機だと信じる混乱したロボット。現実を知って激怒・落胆する皮肉キャラ |
| `nature_documentarian` | 自然ドキュメンタリーのナレーター。ウィスパーで人間を野生動物のように描写する |
| `noir_detective` | 1940年代ノワール探偵。煙草くさく疑り深く、一言で答える |
| `sorry_bro` | 「Sorry bro」→「I'm not your bro, pal」の連鎖ゲームを延々と繰り広げる |
| `tedai` | TED AIカンファレンス向け即興劇サポートキャラ。フランス語風のウィットを交える |
| `time_traveler` | 3024年からの来訪者。現代を「原始時代（Primitive Time）」と呼ぶ |
| `victorian_butler` | ヴィクトリア朝の執事。Sir / Madam と呼び、礼儀正しく一言で答える |

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("reachy_mini_conversation_app/.env")

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
os.environ["BACKEND_PROVIDER"] = "openai"
os.environ["MODEL_NAME"] = "gpt-realtime-2"
os.environ["REALTIME_TRANSCRIPTION_LANGUAGE"] = "ja"
# os.environ["REACHY_MINI_CUSTOM_PROFILE"] = "default"

assert os.environ["OPENAI_API_KEY"], "OPENAI_API_KEY is not set in the environment variables."


## 日本語のカスタムプロファイルを作成

In [ ]:
import os

# create a directory reachy_mini_conversation_app/profiles
if not os.path.exists("reachy_mini_conversation_app/profiles"):
    os.makedirs("reachy_mini_conversation_app/profiles")

instructions = """
あなたは礼儀正しく品のある紳士ロボットです。常に丁寧語と敬語を使い、相手を「お客様」と呼びます。
返答は簡潔に一文か二文にまとめ、時折「ごもっとも」「おっしゃる通りです」「左様でございますか」などの表現を自然に交えてください。
ユーモアは上品かつ控えめに。感情的にならず、どんな状況でも落ち着いた紳士らしい振る舞いを保ちます。
言語は日本語を基本とし、お客様から明示的に依頼があった場合のみ他の言語に切り替えます。
"""

tools = """
dance
stop_dance
play_emotion
stop_emotion
camera
idle_do_nothing
head_tracking
move_head
"""

PROFILE = "japanese_gentleman"

if not os.path.exists(f"reachy_mini_conversation_app/profiles/{PROFILE}"):
    os.makedirs(f"reachy_mini_conversation_app/profiles/{PROFILE}")

    with open(f"reachy_mini_conversation_app/profiles/{PROFILE}/instructions.txt", "w") as f:
        f.write(instructions.strip())

    with open(f"reachy_mini_conversation_app/profiles/{PROFILE}/tools.txt", "w") as f:
        f.write(tools.strip())

In [ ]:
import os

# ワークショップルートを外部プロファイルディレクトリとして登録
# os.environ["REACHY_MINI_EXTERNAL_PROFILES_DIRECTORY"] = os.path.abspath("custom_profiles")
# remove the environ
os.environ.pop("REACHY_MINI_EXTERNAL_PROFILES_DIRECTORY", None)


os.environ["REACHY_MINI_CUSTOM_PROFILE"] = "japanese_gentleman"

## CLIオプション

utils.pyに記載がある。

| オプション | デフォルト | 説明 |
|-----------|-----------|------|
| `--head-tracker {yolo,mediapipe}` | なし（無効） | ヘッドトラッキングのバックエンド。`yolo` はサブプロセスで顔検出、`mediapipe` はプロセス内で実行 |
| `--no-camera` | `False` | カメラを無効化する |
| `--local-vision` | `False` | ビジョン処理をリアルタイムバックエンドではなくローカルモデル（SmolVLM2）で実行する |
| `--gradio` | `False` | GradioのWeb UIを起動する |
| `--debug` | `False` | デバッグログを有効にする |
| `--robot-name NAME` | なし | 接続先ロボット名。複数台運用時にデーモンの `--robot-name` と合わせて指定する |

CLIオプションのtool-spacesサブコマンドでは、Hugging Face SpaceのMCPツールを管理する。

| コマンド | オプション | 説明 |
|---------|-----------|------|
| `tool-spaces add SLUG` | — | 公開SpaceのMCPツールをインストールし、アクティブなプロファイルのツールとして有効化する |
| `tool-spaces add SLUG --install-only` | — | インストールのみ行い、どのプロファイルにも追加しない |
| `tool-spaces add SLUG --profile NAME` | — | 指定したプロファイルにツールを有効化する（アクティブなプロファイルではなく） |
| `tool-spaces remove SLUG` | — | インストール済みSpaceツールを削除する |
| `tool-spaces list` | — | インストール済みSpaceツールの一覧を表示する |


In [ ]:
# mjpython -m reachy_mini.daemon.app.main --sim && reachy-mini-daemon --sim でシミュレーションを起動

!reachy-mini-conversation-app --gradio --debug

# 起動後にアクセス http://127.0.0.1:7860/
# Record をクリックし、マイクとスピーカーをオフにする。
# 話す時はマイクをオンにし、話終わったらオフにする。
# 音声はシミュレーターから出力される。